## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [1]:
import platform

%pip install -q "git+https://github.com/huggingface/optimum-intel.git"
%pip install -qU "openvino>=2025.2" "openvino_tokenizers>=2025.2"

if platform.system() == "Darwin":
    %pip install -q "numpy<2.0.0"

^C
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import requests
from pathlib import Path

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

## Select model
[back to top ⬆️](#Table-of-contents:)

Qwen3 Embedding Series Model list:

| Model Type       | Models               | Size | Layers | Sequence Length | Embedding Dimension | MRL Support | Instruction Aware |
|------------------|----------------------|------|--------|-----------------|---------------------|-------------|----------------|
| Text Embedding   | [Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) | 0.6B | 28     | 32K             | 1024                | Yes         | Yes            |
| Text Embedding   | [Qwen3-Embedding-4B](https://huggingface.co/Qwen/Qwen3-Embedding-4B)   | 4B   | 36     | 32K             | 2560                | Yes         | Yes            |
| Text Embedding   | [Qwen3-Embedding-8B](https://huggingface.co/Qwen/Qwen3-Embedding-8B)   | 8B   | 36     | 32K             | 4096                | Yes         | Yes            |

In [2]:
import ipywidgets as widgets

##model_id = "Qwen/Qwen3-4B-Instruct-2507"
model_id = "ibm-granite/granite-4.0-micro"

## Convert model using Optimum Intel
[back to top ⬆️](#Table-of-contents:)

For convenience, we will use OpenVINO integration with HuggingFace Optimum. 🤗 [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) is the interface between the 🤗 Transformers and Diffusers libraries and the different tools and libraries provided by Intel to accelerate end-to-end pipelines on Intel architectures.

Among other use cases, Optimum Intel provides a simple interface to optimize your Transformers and Diffusers models, convert them to the OpenVINO Intermediate Representation (IR) format and run inference using OpenVINO Runtime. `optimum-cli` provides command line interface for model conversion and optimization. 

General command format:

```bash
optimum-cli export openvino --model <model_id_or_path> --task <task> <output_dir>
```

where task is task to export the model for, if not specified, the task will be auto-inferred based on the model. You can find a mapping between tasks and model classes in Optimum TaskManager [documentation](https://huggingface.co/docs/optimum/exporters/task_manager). Additionally, you can specify weights compression using `--weight-format` argument with one of following options: `fp32`, `fp16`, `int8` and `int4`. Fro int8 and int4 [nncf](https://github.com/openvinotoolkit/nncf) will be used for  weight compression. More details about model export provided in [Optimum Intel documentation](https://huggingface.co/docs/optimum/intel/openvino/export#export-your-model).

In [3]:
to_compress = widgets.Checkbox(
    value=False,
    description="Weight compression",
    disabled=False,
)

visible_widgets = [to_compress]

options = widgets.VBox(visible_widgets)

options

The Qwen3-embedding model can be exported by `feature-extraction` task with Optimum-intel.

In [4]:
from pathlib import Path

#model_base_dir = Path(model_id.split("/")[-1])
model_base_dir = Path("granite-4.0-micro")
additional_args = {"task": "text-generation"}


model_dir = model_base_dir / "FP16"
additional_args.update({"weight-format": "fp16"})

In [5]:
from cmd_helper import optimum_cli

if not model_dir.exists():
    optimum_cli(model_id, model_dir, additional_args=additional_args)

## Run OpenVINO model inference with Optimum-intel
[back to top ⬆️](#Table-of-contents:)

Select device from dropdown list for running inference using OpenVINO.

In [6]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU"])
device

Dropdown(description='Device:', options=('CPU', 'GPU.0', 'GPU.1', 'AUTO'), value='CPU')

The Qwen3-embedding model can be exported by class `OVModelForFeatureExtraction` with Optimum-intel.

In [7]:
from optimum.intel import OVModelForCausalLM

model = OVModelForCausalLM.from_pretrained(model_dir, device=device.value, export=False, use_cache = False)

`dynamic_shapes` was set to True, but this value will be ignored as only dynamic shapes are supported.


In [8]:
import torch
import torch.nn.functional as F

from torch import Tensor
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code = True)

model.compile(
)

messages = [
    {"role": "system", "content": "Eres un asistente útil, claro y conciso."},
    {"role": "user", "content": "Explícame qué es la gravedad en dos frases."}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=16,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.05
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

systemEres un asistente útil, claro y conciso.
userExplícame qué es la gravedad en dos frases.
assistantLa gravedad es una fuerza fundamental que atrae objetos con masa


In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=16,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.05
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))